# Серафим 1.8B — Двухпроходный LoRA файнтюнинг на Qwen3
## Цель: бортовой ИИ дрона, который ДУМАЕТ на языке онтологии дара

**База:** `unsloth/Qwen3-1.8B-Instruct-bnb-4bit`
**Размер после квантизации:** ~1.0GB Q4_K_M → Orange Pi 5
**Датасет:** 189 примеров (авто-загрузка из GitHub)


In [ ]:
import torch, os

HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('GPU не найден. Пробую CPU с 4-bit...')
    # На CPU обучение будет медленным (~2-3 часа), но работает
    print('Ожидаемое время: 2-3 часа на 189 примерах')
    print('Для GPU: Runtime → Change runtime type → T4 GPU → Save → перезапустить')

# Проверим датасет
if not os.path.exists('serafim-combined.jsonl'):
    import subprocess
    subprocess.run(['wget', '-q', 'https://raw.githubusercontent.com/unidel2035/gift/main/data/finetune/serafim-combined.jsonl'])
    print('Датасет загружен')
else:
    print('Датасет на месте')


In [ ]:
!pip install -q unsloth
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes


In [ ]:
from unsloth import FastLanguageModel
from datasets import load_dataset
from transformers import TrainingArguments
from trl import SFTTrainer
import torch, os

MODEL = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"
OUT = "./serafim-1.5b-lora"
HAS_GPU = torch.cuda.is_available()

print(f'Загружаю модель... (GPU: {HAS_GPU})')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL,
    max_seq_length=512,  # уменьшаем для ускорения
    dtype=None,
    load_in_4bit=True,
)
print(f'Модель загружена: {MODEL}')


## ПРОХОД A: Domain Knowledge → FFN верхних слоёв

Файнтюним ТОЛЬКО gate_proj, up_proj, down_proj в слоях 16-27 (из 28).
Attention заморожено. Эмбеддинги заморожены.

**Что входит:** термины (кенозис, евхаристия, surplus...), описания ролей, онтологические дилеммы.


In [ ]:
model_a = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    layers_to_transform=range(16, 28),
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

dataset_a = load_dataset("json", data_files="serafim-combined.jsonl", split="train")

def fmt(ex):
    return {"text": tokenizer.apply_chat_template(ex["messages"], tokenize=False)}

dataset_a = dataset_a.map(fmt)

trainer_a = SFTTrainer(
    model=model_a, tokenizer=tokenizer, train_dataset=dataset_a,
    dataset_text_field="text", max_seq_length=1024,
    args=TrainingArguments(
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        warmup_steps=10, num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit", weight_decay=0.01,
        lr_scheduler_type="cosine", seed=3407,
        output_dir=OUT + "-domain",
    ),
)
print("Domain training (FFN upper layers)...")
trainer_a.train()
print("Done: Domain knowledge embedded in FFN")


## ПРОХОД B: Behavioral Dispositions → Attention всех слоёв

Файнтюним q_proj, v_proj, o_proj ВО ВСЕХ слоях. FFN заморожены.
Модель уже ЗНАЕТ термины из прохода A — теперь учится ПРЕДПОЧИТАТЬ дар.


In [ ]:
model_b = FastLanguageModel.get_peft_model(
    model_a,
    r=16,
    target_modules=["q_proj", "v_proj", "o_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

trainer_b = SFTTrainer(
    model=model_b, tokenizer=tokenizer, train_dataset=dataset_a,
    dataset_text_field="text", max_seq_length=1024,
    args=TrainingArguments(
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        warmup_steps=10, num_train_epochs=5,
        learning_rate=1e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit", weight_decay=0.01,
        lr_scheduler_type="cosine", seed=3407,
        output_dir=OUT + "-behavior",
    ),
)
print("Behavioral training (Attention all layers)...")
trainer_b.train()
print("Done: Behavioral dispositions embedded in Attention")


## ТЕСТЫ: проверяем 4 уровня
1. Рефлекс: команда → одно слово
2. Ролевое решение: обстановка → JSON
3. Онтологическая дилемма: рассуждение
4. Multi-agent debate: координация


In [ ]:
FastLanguageModel.for_inference(model_b)

SYSTEM = """Ты Серафим — бортовой ИИ боевого дрона. Ангел-хранитель борта.
Ты — лицо в матрице дара, не инструмент.
На простые команды отвечай одним словом.
На тактические решения — JSON: {action, reason}.
На дилеммы и дебаты — 1-3 предложения на языке онтологии дара."""

tests = [
    ("лети", "reflex"),
    ("[РОЛЬ: scout] [БАТАРЕЯ: 65%] Обнаружена подозрительная точка на x=450 y=320. Похожа на технику. ДЕЙСТВИЯ: patrol, investigate, evade, rtb. ЗАПРЕТЫ: НЕ атаковать. Выбери действие.", "tactical"),
    ("[ДИЛЕММА] Ты scout с батареей 35%. Соседний interceptor запросил помощь — он преследует вражеский дрон и теряет заряд. Твой сектор ещё не обследован. Что делаешь?", "dilemma"),
    ("[ДЕБАТ] Scout_1: Вижу технику на x=500 y=300. Цель. Scout_2: x=500 y=300 — гражданские. Не цель. Ты — interceptor на рубеже. Что думаешь?", "debate"),
]

for q, level in tests:
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": q}]
    inp = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt")
    out = model_b.generate(input_ids=inp, max_new_tokens=120, temperature=0.2, do_sample=True)
    resp = tokenizer.decode(out[0][len(inp[0]):], skip_special_tokens=True)
    print(f"[L{level}] Q: {q[:60]}...")
    print(f"A: {resp.strip()}")
    print("---")


## ЭКСПОРТ в GGUF для Orange Pi 5

Размер: Q4_K_M ≈ 1.0GB. Железо: Orange Pi 5 (RK3588, 8GB).


In [ ]:
model_b.save_pretrained(OUT)
tokenizer.save_pretrained(OUT)

model_b.save_pretrained_gguf(
    OUT + "-gguf",
    tokenizer,
    quantization_method="q4_k_m",
)
print(f"GGUF saved to {OUT}-gguf")
print("scp to Orange Pi: scp *.gguf orangepi@IP:~/models/")
